# 03 — URL preprocessing và feature engineering
**Phụ trách:** Nguyễn Khang Hy  
**Trạng thái:** template, chưa triển khai parser/extractor.

**Đầu ra:** parser, domain đăng ký, bộ features có định nghĩa và ma trận X gắn id. Tập 21 ứng viên là tham khảo schema, không phải bộ tối ưu đã xác nhận.

## 0. Môi trường và đường dẫn
Mặc định chạy local: cài requirements.txt trong .venv, mở notebook bằng VS Code/Jupyter và chọn kernel của .venv. Dataset ở data/ hoặc vị trí PHISHGUARD_DATA_DIR trên máy mình. Ô dưới chỉ nạp đường dẫn/config; Colab là lựa chọn thêm theo README.

In [1]:
from pathlib import Path
import os
import sys
import tomllib

start = Path(os.environ.get("PHISHGUARD_ROOT", Path.cwd())).expanduser().resolve()
candidates = (start, *start.parents)
PROJECT_ROOT = next(
    (p for p in candidates if (p / "pyproject.toml").is_file()
     and (p / "src" / "phishguard_vn").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Chuyen working directory vao repo hoac dat PHISHGUARD_ROOT "
        "toi thu muc du an tren may ban."
    )
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from phishguard_vn.paths import get_data_dir, get_config_path
DATA_DIR = get_data_dir()
with get_config_path().open("rb") as handle:
    CONFIG = tomllib.load(handle)

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("Config:", CONFIG["experiment"]["name"])


Project: /Users/khangcero/PhishGuardVN
Dataset: /Users/khangcero/PhishGuardVN/data
Config: week4_e1_core_lr


## 1. Parser và quy ước chuẩn hóa
TODO: triển khai url_parser.py; giữ raw_url, tách domain bằng Public Suffix List; xử lý thiếu scheme, IP, @, port, Unicode/Punycode và path/query. Ghi phiên bản PSL và quy tắc private suffix.

In [ ]:
from phishguard_vn.url_parser import parse_url
from phishguard_vn.features import CANDIDATE_FEATURES, extract_features, build_feature_table

print("Reference candidate count:", len(CANDIDATE_FEATURES))

# Test xem parser chạy đúng không
test_url = "https://login-bank.com/verify?id=123"
parsed = parse_url(test_url)

print("Kết quả bóc tách URL:")
print(parsed)


Reference candidate count: 12
Kết quả bóc tách URL:
ParsedURL(raw_url='https://login-bank.example/verify?id=123', hostname='login-bank.example', registrable_domain=None, subdomain='login-bank', path='/verify', query='id=123', port=None, scheme_was_added=False, is_ip=False, is_valid=False, error_reason='unresolved_domain')


## 2. Định nghĩa và trích xuất features
TODO: tham khảo docs/schema.md và vn_compphish.csv; xác định công thức/phạm vi đếm. Điền feature_dictionary.csv. Loại is_https và metadata khỏi X; ghi lý do giữ/bỏ cột.
Ghi chú Schema (P1): Đã chọn lọc 12/21 lexical features từ schema.md. Loại bỏ các features tỷ lệ (ratios) do dư thừa tuyến tính khi đã có tổng chiều dài và số đếm. Các query features (eq_cnt, amp_cnt) tạm bỏ để làm nhẹ baseline do dataset hiện tại ít query strings.

In [ ]:
from phishguard_vn.features import CANDIDATE_FEATURES, extract_features

print("Danh sách 12 Features đã chốt:", CANDIDATE_FEATURES)

# Test thử tính toán đặc trưng
feats = extract_features("https://login-bank.com/verify?id=123")
print("\nKết quả trích xuất Features:")
print(feats)

Danh sách 12 Features đã chốt: ('url_len', 'dom_len', 'is_ip', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'slash_cnt', 'entropy', 'path_len')

Kết quả trích xuất Features:
{'url_len': None, 'dom_len': None, 'is_ip': None, 'subdom_cnt': None, 'letter_cnt': None, 'digit_cnt': None, 'dot_cnt': None, 'dash_cnt': None, 'under_cnt': None, 'slash_cnt': None, 'entropy': None, 'path_len': None}


## 3. Kiểm tra các trường hợp dễ sai
TODO: ít nhất 10 ví dụ tổng hợp gồm thiếu scheme, domain nhiều cấp, IP, @, port lỗi, IDN, hostname hoa/thường, path/query giữ case, subdomain và đầu vào không hợp lệ.

In [4]:
from phishguard_vn.url_parser import parse_url

# (url, registrable_domain, is_valid, is_ip, error_reason)
CASES = [
    ("google.com",                         "google.com",       True,  False, None),
    ("HTTPS://WWW.Example.COM/Path?Q=A",   "example.com",      True,  False, None),
    ("a.b.c.example.co.uk/x",              "example.co.uk",    True,  False, None),
    ("example.com:8080/a",                 "example.com",      True,  False, None),
    ("http://paypal.com@evil.com/login",   "evil.com",         True,  False, None),
    ("https://xn--mnchen-3ya.de/",         "xn--mnchen-3ya.de", True, False, None),
    ("http://192.168.1.1/login",           None,               True,  True,  "is_ip_no_domain"),
    ("http://[2001:db8::1]/",              None,               True,  True,  "is_ip_no_domain"),
    ("https://example.com:99999/",         None,               False, False, "invalid_port"),
    ("invalid_url",                        None,               False, False, "unresolved_domain"),
    ("",                                   None,               False, False, "empty_url"),
    (None,                                 None,               False, False, "empty_url"),
]

print("--- CHẠY ASSERTIONS KIỂM THỬ ---")
for url, exp_domain, exp_valid, exp_ip, exp_err in CASES:
    p = parse_url(url)
    got = (p.registrable_domain, p.is_valid, p.is_ip, p.error_reason)
    exp = (exp_domain, exp_valid, exp_ip, exp_err)
    assert got == exp, f"FAIL {url!r}: got={got} expected={exp}"

# IP không bao giờ được làm group key
assert all(parse_url(u).registrable_domain is None for u in ("http://192.168.1.1/", "http://[2001:db8::1]/"))
# Giữ case của path/query, hạ chữ thường hostname
p = parse_url("HTTPS://WWW.Example.COM/Path?Q=A")
assert p.path == "/Path" and p.query == "Q=A"

print(f"Tất cả {len(CASES)} test cases pass!")

--- CHẠY ASSERTIONS KIỂM THỬ ---
Tất cả 12 test cases pass!


## 4. Ma trận features và bàn giao
TODO: tạo X giữ id, kiểm tra số dòng, NaN/inf, thứ tự cột và cột hằng trên train. Domain bàn giao cho Hưng; X/extractor bàn giao cho Phúc. Fit scaler/selector là việc bên trong pipeline train/fold.

In [5]:
import numpy as np
import pandas as pd
from phishguard_vn.features import build_feature_table, CANDIDATE_FEATURES

ID_COL = "id"
FEATURE_COLS = list(CANDIDATE_FEATURES)
assert len(FEATURE_COLS) == 12

input_path = PROJECT_ROOT / "data" / "interim" / "core_gold_silver.csv"
frame = pd.read_csv(input_path)
print(f"Đang trích xuất features cho {len(frame)} dòng...")

feature_table = build_feature_table(frame)
out_dir = PROJECT_ROOT / "data" / "interim"
out_dir.mkdir(parents=True, exist_ok=True)

# Metadata cho Hưng
domain_meta = feature_table[[ID_COL, "registrable_domain", "error_reason"]].copy()
domain_meta.to_csv(out_dir / "03_registrable_domains.csv", index=False)

# Ma trận X cho Phúc (Chỉ ID + 12 features)
matrix_X = feature_table[[ID_COL, *FEATURE_COLS]].copy()
assert list(matrix_X.columns) == [ID_COL, *FEATURE_COLS], "Sai cột/thứ tự của X"
assert "error_reason" not in matrix_X.columns
assert "registrable_domain" not in matrix_X.columns
matrix_X.to_csv(out_dir / "03_matrix_X_features.csv", index=False)

print("-> Đã xuất 03_registrable_domains.csv và 03_matrix_X_features.csv")

Đang trích xuất features cho 18997 dòng...
-> Đã xuất 03_registrable_domains.csv và 03_matrix_X_features.csv


In [ ]:
print("=" * 60)
print("HANDOFF CHECK")
print("=" * 60)
problems = []

# 1. Số dòng
n_in, n_x, n_d = len(frame), len(matrix_X), len(domain_meta)
print(f"[rows] input={n_in} | X={n_x} | domain={n_d}")
if not (n_in == n_x == n_d):
    problems.append("Số dòng không khớp")

# 2. ID trùng / thiếu / thừa
dup_x = int(matrix_X[ID_COL].duplicated().sum())
dup_in = int(frame[ID_COL].duplicated().sum())
missing = set(frame[ID_COL]) - set(matrix_X[ID_COL])
extra = set(matrix_X[ID_COL]) - set(frame[ID_COL])
print(f"[id] trùng(X)={dup_x} | trùng(input)={dup_in} | thiếu={len(missing)} | thừa={len(extra)}")
if dup_x or missing or extra:
    problems.append("ID trùng/thiếu/thừa")

# 3. NaN / Inf
vals = matrix_X[FEATURE_COLS].to_numpy(dtype=float)
nan_total = int(np.isnan(vals).sum())
inf_total = int(np.isinf(vals).sum())
rows_nan = int(np.isnan(vals).any(axis=1).sum())
print(f"[values] NaN={nan_total} (trên {rows_nan} dòng) | Inf={inf_total}")
if inf_total:
    problems.append("Có Inf trong X")

# 4. Domain chưa xác định
dom_na = domain_meta["registrable_domain"].isna()
print(f"[domain] chưa xác định={int(dom_na.sum())}/{n_d}")
print("   phân loại error_reason:")
print(domain_meta.loc[dom_na, "error_reason"].value_counts(dropna=False).to_string())

# 5. Số URL đủ điều kiện Group CV
eligible = domain_meta[~dom_na]
print(f"[group-cv] URL đủ điều kiện={len(eligible)}/{n_d} ({len(eligible)/n_d:.2%}) | số domain duy nhất={eligible['registrable_domain'].nunique()}")

# 6. Cột hằng
const_cols = [c for c in FEATURE_COLS if matrix_X[c].nunique(dropna=True) <= 1]
print(f"[const] cột hằng: {const_cols or 'không có'}")
print("-" * 60)

# --- BỔ SUNG AUDIT CHI TIẾT ---
bad = domain_meta.loc[dom_na, [ID_COL]].merge(frame[[ID_COL, "url"]], on=ID_COL)
print("\n[domain] URL chưa xác định:")
print(bad.to_string(index=False))

top = eligible["registrable_domain"].value_counts().head(10)
print(f"\n[group-cv] top 10 domain (tổng {top.sum()} URL, {top.sum()/len(eligible):.2%}):")
print(top.to_string())

if "label" in frame.columns:
    print("\n[label]", frame.set_index(ID_COL).loc[eligible[ID_COL], "label"].value_counts().to_dict())
# ------------------------------

if problems:
    raise AssertionError("HANDOFF FAIL: " + "; ".join(problems))
print("\nHANDOFF OK: có thể bàn giao.")

HANDOFF CHECK
[rows] input=18997 | X=18997 | domain=18997
[id] trùng(X)=0 | trùng(input)=0 | thiếu=0 | thừa=0
[values] NaN=24 (trên 2 dòng) | Inf=0
[domain] chưa xác định=2/18997
   phân loại error_reason:
error_reason
unresolved_domain    2
[group-cv] URL đủ điều kiện=18995/18997 (100.0%) | số domain duy nhất=12277
[const] cột hằng: ['is_ip']
------------------------------------------------------------
HANDOFF OK: có thể bàn giao.


In [ ]:
ipv4_like = frame["url"].astype(str).str.contains(
    r"^(?:[a-zA-Z][a-zA-Z0-9+.-]*://)?\d{1,3}(?:\.\d{1,3}){3}(?:[:/]|$)", regex=True
)
print("URL dạng IPv4 trong dữ liệu:", int(ipv4_like.sum()))

## 5. Kết luận
- Đã triển khai thành công module `url_parser` bóc tách domain bằng thư viện `tldextract` (Public Suffix List).
- Đã xử lý các trường hợp ngoại lệ: URL thiếu scheme, IP, port, @, Punycode.
- Đã trích xuất 12 features số cơ bản. Đã loại bỏ `is_https` và các metadata theo kế hoạch để tránh rò rỉ dữ liệu.
- Đã xuất file `registrable_domain` cho Hưng và ma trận `X` cho Phúc.